# Statim: licence-clean training on an A100 (one click, three seeds)

**Run: Runtime → Run all (Ctrl+F9).** The A100 is preselected. The secret `HF_TOKEN` must allow notebook access.

The 0.10.0 recipe, with only data and start point changed, so the weights can be offered for commercial use:
- **Start:** `convaiinnovations/laya-multilingual` (Apache-2.0, mmBERT-base). Not 0.4.0, whose training data was not cleared.
  Distillation also learns only from this clean start model.
- **Data:** `mixture-v9` from `tools/finetune/build_v9.py`, the sources that passed the two-step licence audit
  (sha256 bba18941…), plus Banking77, MASSIVE and typed decisions.
- **Three seeds in parallel** on one GPU, without gradient checkpointing. Each finished run uploads to the
  **private** repo `Beko2210/statim-train-staging` under `runs/`. Soup and gate run on pop-os. Nothing is published.


In [ ]:
# ===== SETUP (~5 min) =====
import os, subprocess, sys, time, json, threading
PREFIX = "v12-clean"
SEEDS = [1, 2, 3]
STAGING = "Beko2210/statim-train-staging"          # private
BASE = "/content/stage/base/laya-multilingual"
MIXTURE = "/content/stage/data/mixture-v9.jsonl.gz"
BUDGET = "banking77=12000,massive=24000,mixture=96000,typed=6000,distill=18000"

def sh(cmd):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.returncode:
        print(r.stdout[-3000:], r.stderr[-3000:]); raise SystemExit("FAILED: " + cmd)
    return r.stdout

print("GPU:", sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader").strip())
subprocess.run("pkill -f 'vllm serve'; sleep 5", shell=True)
from google.colab import userdata
os.environ["HF_TOKEN"] = HF_TOKEN = userdata.get("HF_TOKEN")
sh("pip -q install laya==0.3.20 datasets huggingface_hub safetensors")
os.chdir("/content")
sh("rm -rf /content/statim && git clone -q --depth 1 -b v0.10.0 https://github.com/BEKO2210/statim /content/statim")
os.chdir("/content/statim")
print(sh("git log --oneline -1"))
from huggingface_hub import HfApi, snapshot_download
api = HfApi(token=HF_TOKEN)
snapshot_download(STAGING, token=HF_TOKEN, local_dir="/content/stage",
                  allow_patterns=["base/laya-multilingual/*", "base/laya-multilingual/**", "data/mixture-v9.jsonl.gz"])
import hashlib
print("mixture sha256:", hashlib.sha256(open(MIXTURE, "rb").read()).hexdigest()[:16])
print("SETUP OK")


In [ ]:
# ===== TRAIN three seeds in parallel (~5-6 h; each run uploads itself when done) =====
def run(seed):
    name = f"{PREFIX}-s{seed}"
    out, log = f"/content/{name}", f"/content/{name}.log"
    cmd = [sys.executable, "-u", "tools/finetune/train_multitask.py", BASE, out, "--clean", "--mixture", MIXTURE,
           "--mixture-dev", "6000", "--massive-per-lang", "11000", "--max-len", "1024", "--epochs", "12",
           "--patience", "3", "--distill", "24000", "--warmup", "0.06", "--ema", "0.999", "--budget", BUDGET,
           "--max-tokens", "16384", "--accum", "1", "--train-seed", str(seed), "--no-grad-ckpt", "--tag", name]
    with open(log, "w") as f:
        code = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT).returncode
    for attempt in range(3):
        try:
            api.upload_file(path_or_fileobj=log, path_in_repo=f"runs/{name}/train.log", repo_id=STAGING,
                            commit_message=f"{name}: log (exit {code})")
            if code == 0:
                api.upload_folder(folder_path=out, path_in_repo=f"runs/{name}/model", repo_id=STAGING,
                                  commit_message=f"{name}: trained model")
            break
        except Exception as exc:
            print(name, "upload failed, retrying:", type(exc).__name__, flush=True); time.sleep(60)
    print(f"{name}: exit {code}, uploaded", flush=True)

threads = [threading.Thread(target=run, args=(s,)) for s in SEEDS]
for t in threads: t.start()
seen = {}
while any(t.is_alive() for t in threads):
    time.sleep(60)
    for s in SEEDS:   # live view: every finished epoch of every seed
        p = f"/content/{PREFIX}-s{s}.log"
        if os.path.exists(p):
            for line in open(p, errors="replace").read().replace("\r", "\n").splitlines():
                if (line.startswith("=== epoch") or line.startswith("saved") or "Traceback" in line) and line not in seen.setdefault(s, set()):
                    seen[s].add(line); print(f"[seed {s}] {line[:160]}", flush=True)
for t in threads: t.join()
print("ALL DONE. Next: soup and gate on pop-os.")
